# Transformer for Language Identification


In this notebook, we implement a language classifier based on the **Transformer (encoder-only)** architecture, built piece by piece. The dataset used is **`papluca/language-identification`**, the same one used to train the LSTM model. The final goal is to compare both architectures on the same task and the same data.

### Why use the same dataset as the LSTM?
Using the same dataset enables a direct and fair comparison: same texts, same labels, same class distribution. Any metric differences can be attributed to the architecture rather than task difficulty.

#### References
- Dataset: https://huggingface.co/datasets/papluca/language-identification
- [Attention is All You Need](http://arxiv.org/abs/1706.03762)
- [NLP with Transformers — Tunstall et al.](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [Tutorial 5: Transformers and Multi-Head Attention — Lightning AI](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/05-transformers-and-MH-attention.html)

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages
print(f'Running in Colab: {IN_COLAB}')

In [ ]:
# Instalar dependencias si estamos en Colab
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]'==4.41.2 pytorch-lightning torchmetrics

## 1. Dataset Loading

We load `papluca/language-identification`. This dataset contains text snippets in **20 languages**, each labeled with its ISO language code.

This is the **same dataset** used in the LSTM notebook, which allows a direct architecture comparison at the end.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

dataset = load_dataset('papluca/language-identification', split='train')
print(dataset)

In [ ]:
print('Sample record:')
dataset[0]

### 1.1 Language Distribution

We explore how many samples there are per language. A balanced dataset helps avoid training bias.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

df_raw = pd.DataFrame(dataset)
lang_counts = df_raw['labels'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(lang_counts.index, lang_counts.values, color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%d', fontsize=8, padding=2)
ax.set_title('Sample Distribution by Language (papluca/language-identification)', fontsize=13)
ax.set_xlabel('Language (ISO 639-1 code)')
ax.set_ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lang_distribution.png', dpi=120)
plt.show()

print(f'Total languages: {lang_counts.shape[0]}')
print(lang_counts.to_string())

**Note:** The dataset is perfectly balanced: every language has exactly the same number of samples. This is beneficial because no oversampling or class weighting techniques are needed.

### 1.2 Text Length Statistics

In [ ]:
text_lengths = [len(row['text']) for row in dataset]
print(f'Shortest text:  {min(text_lengths)} characters')
print(f'Longest text:   {max(text_lengths)} characters')
print(f'Average:        {np.mean(text_lengths):.1f} characters')
print(f'Median:         {np.median(text_lengths):.1f} characters')
print(f'95th percentile:{np.percentile(text_lengths, 95):.1f} characters')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(text_lengths, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(text_lengths), color='red', linestyle='--',
                label=f'Mean: {np.mean(text_lengths):.0f}')
axes[0].axvline(np.percentile(text_lengths, 95), color='orange', linestyle='--',
                label=f'P95: {np.percentile(text_lengths, 95):.0f}')
axes[0].set_title('Length Distribution (all texts)')
axes[0].set_xlabel('Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist([l for l in text_lengths if l < 500], bins=60, color='coral', edgecolor='white')
axes[1].set_title('Length Distribution (< 500 characters)')
axes[1].set_xlabel('Length (characters)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('text_length_distribution.png', dpi=120)
plt.show()

**Note:** Most texts are short (under 200 characters). The LSTM used `max_length=128` tokens, so we adopt the same value to keep the models comparable.

## 2. Tokenizer Definition

We use the **same simple word-level tokenizer** from the LSTM notebook. This keeps things consistent: both models process text the same way, so any performance differences come only from the architecture.

The tokenizer:
1. Converts text to lowercase.
2. Replaces non-alphabetic characters with spaces.
3. Splits on spaces.
4. Maps each token to an integer ID (or `[UNK]` if it is out of vocabulary).

In [ ]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    cleaned = [c if c.isalpha() else ' ' for c in text]
    cleaned_text = re.sub(r'\s+', ' ', ''.join(cleaned)).strip()
    return cleaned_text.split()

# Build vocabulary
token_counts = Counter()
for text in dataset['text']:
    token_counts.update(simple_tokenizer(text))

top_n_tokens = list(token_counts.keys())[:50000 - 2]
vocab = {'[PAD]': 0, '[UNK]': 1}
vocab.update({token: idx + 2 for idx, token in enumerate(top_n_tokens)})

def tokenize_text(text, max_length=128):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(t, vocab['[UNK]']) for t in tokens]
    ids = ids[:max_length]
    ids = ids + [vocab['[PAD]']] * (max_length - len(ids))
    return ids

print(f'Vocabulary size: {len(vocab)} tokens')
print(f'First 15 tokens: {top_n_tokens[:15]}')
print(f'Middle 15 tokens: {top_n_tokens[1000:1015]}')
print(f'Last 15 tokens:  {top_n_tokens[-15:]}')

In [ ]:
# Quick check
test_sentence = 'Hello World in many languages'
tokenized = tokenize_text(test_sentence, max_length=8)
id_2_token = {v: k for k, v in vocab.items()}
print(f'Sentence: "{test_sentence}"')
print(f'IDs:     {tokenized}')
print(f'Tokens:  {[id_2_token[t] for t in tokenized]}')

### 2.1 Vocabulary by Language

How many unique tokens does each language contribute? This shows which languages are lexically richer in this dataset and which may be harder for the simple tokenizer.

In [ ]:
lang_vocab_sizes = {}
for lang in df_raw['labels'].unique():
    texts = df_raw[df_raw['labels'] == lang]['text'].tolist()
    tokens = set(t for text in texts for t in simple_tokenizer(text))
    lang_vocab_sizes[lang] = len(tokens)

lang_vocab_df = pd.Series(lang_vocab_sizes).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(lang_vocab_df.index, lang_vocab_df.values, color='teal', edgecolor='white')
ax.bar_label(bars, fmt='%d', fontsize=8, padding=3)
ax.set_title('Unique Tokens by Language', fontsize=13)
ax.set_xlabel('Number of Unique Tokens')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('vocab_per_lang.png', dpi=120)
plt.show()

print('\nTop 5 (most unique tokens):')
print(lang_vocab_df.head(5).to_string())

**Note:** Languages with non-Latin scripts (Arabic, Russian, Chinese, Japanese) contain many unique tokens that a simple word-level tokenizer may handle less efficiently, since keeping only alphabetic characters can collapse parts of those scripts. A subword tokenizer (BPE, SentencePiece) would be more appropriate for them.

## 3. PyTorch Dataset

In [ ]:
import torch
from typing import Dict
from torch.utils.data import Dataset

class LanguageIdentificationDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 128):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['labels'])))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text  = self.dataset[index]['text']
        label = self.dataset[index]['labels']
        y = self.class_2_id_map[label]
        input_ids = self.tokenizer(text, max_length=self.seq_length)
        # Attention mask: 1 = real token, 0 = padding
        attention_mask = [1 if tok != vocab['[PAD]'] else 0 for tok in input_ids]
        return {
            'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'y':              torch.tensor(y,              dtype=torch.long)
        }

max_len = 128  # Same as the LSTM for comparability
lang_dataset = LanguageIdentificationDataset(tokenize_text, dataset, seq_length=max_len)

assert len(lang_dataset) == len(dataset)
print(f'Dataset instanciado: {len(lang_dataset)} muestras')
print(f'Number of classes: {lang_dataset.num_classes}')
print(f'Languages:           {list(lang_dataset.id_2_class_map.values())}')
sample = lang_dataset[0]
print(f"\nEjemplo:")
print(f"  input_ids shape:      {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  y: {sample['y']} ({lang_dataset.id_2_class_map[sample['y'].item()]})")

In [ ]:
from torch.utils.data import random_split, DataLoader

batch_size = 4 if not IN_COLAB else 128

train_dataset, val_dataset, test_dataset = random_split(
    lang_dataset, lengths=[0.8, 0.1, 0.1],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Batch size: {batch_size}')

## 4. Transformer Architecture

We build the architecture piece by piece following *"Attention is All You Need"* (Vaswani et al., 2017).

### 4.1 Positional Encoding

The Transformer has no recurrence or convolution, so it does not naturally encode token order. **Sinusoidal positional encoding** injects that information:

$$
PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) \qquad
PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

In [ ]:
import torch.nn as nn
from enum import Enum


class PosEncodingType(Enum):
    SINUSOID  = 1
    LEARNABLE = 2


class SinusoidPE(nn.Module):
    """Positional Encoding sinusoidal (pre-calculado, no entrenable)."""

    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        pos      = torch.arange(max_len).unsqueeze(1)   # (max_len, 1)
        i        = torch.arange(d_model).unsqueeze(0)   # (1, d_model)
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        angle_rads = pos * div_term
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        pe[:, 1::2] = torch.cos(angle_rads[:, 1::2])
        self.register_buffer('pos_encoding', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pos_encoding[:, :x.size(1)]


class LearnablePE(nn.Module):
    """Positional Encoding aprendido durante entrenamiento."""

    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)


class TokenAndPosEmbedding(nn.Module):
    """Combina embedding de tokens con positional encoding."""

    def __init__(self, max_len: int, d_model: int, vocab_size: int,
                 pe_type: PosEncodingType = PosEncodingType.SINUSOID):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb   = SinusoidPE(max_len, d_model) if pe_type == PosEncodingType.SINUSOID else LearnablePE(max_len, d_model)
        self.dropout   = nn.Dropout(0.1)

    def forward(self, x):
        return self.dropout(self.pos_emb(self.token_emb(x)))


print('Módulos de embedding definidos.')

#### Positional Encoding Visualization

We plot the PE matrix to understand what positional information the model receives.

In [ ]:
emb_dim = 128 if not IN_COLAB else 256

tpe = TokenAndPosEmbedding(max_len, emb_dim, len(vocab))
pe_matrix = tpe.pos_emb.pos_encoding.squeeze(0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im = axes[0].pcolormesh(pe_matrix, cmap='RdBu')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position in sequence')
axes[0].set_title('Positional Encoding Sinusoidal')
plt.colorbar(im, ax=axes[0])

for pos in [0, 10, 30, 63]:
    axes[1].plot(pe_matrix[pos, :], label=f'pos={pos}')
axes[1].set_xlabel('Embedding Dimension')
axes[1].set_ylabel('Value')
axes[1].set_title('PE Values for Specific Positions')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=120)
plt.show()
print('PE matrix shape:', pe_matrix.shape)

**Note:** The first dimensions oscillate quickly (high frequency) and the last ones slowly (low frequency). This allows the model to distinguish nearby and distant positions smoothly and continuously, without training extra parameters.

### 4.2 Multi-Head Attention

The core of the Transformer. **Scaled Dot-Product Attention** is defined as:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_K}}\right)V
$$

Multi-Head Attention projects Q, K, and V into $h$ subspaces, computes attention in parallel, and concatenates the results.

In [ ]:
import math


class MultiHeadAttention(nn.Module):

    def __init__(self, embed_size: int, num_heads: int = 8):
        super().__init__()
        self.embed_size     = embed_size
        self.num_heads      = num_heads
        assert embed_size % num_heads == 0, 'embed_size debe ser divisible por num_heads'
        self.projection_dim = embed_size // num_heads

        self.query        = nn.Linear(embed_size, embed_size)
        self.key          = nn.Linear(embed_size, embed_size)
        self.value        = nn.Linear(embed_size, embed_size)
        self.combine_heads = nn.Linear(embed_size, embed_size)

    @staticmethod
    def _scaled_dot_product(q, k, v, mask=None):
        """
        q, k, v: (batch, heads, seq_len, proj_dim)
        mask:    (batch, seq_len) — 1 = token real, 0 = padding
        """
        d_k    = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, L, L)
        if mask is not None:
            mask   = mask.unsqueeze(1).unsqueeze(2).float()  # (B, 1, 1, L)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        return torch.matmul(attn_weights, v), attn_weights

    def _split_heads(self, x):
        """(B, L, E) -> (B, H, L, proj_dim)"""
        B, L, _ = x.size()
        return x.view(B, L, self.num_heads, self.projection_dim).permute(0, 2, 1, 3)

    def forward(self, x, mask=None):
        q = self._split_heads(self.query(x))
        k = self._split_heads(self.key(x))
        v = self._split_heads(self.value(x))

        attn_out, self.attn_weights = self._scaled_dot_product(q, k, v, mask)

        # Reensamblar cabezas
        B, _, L, _ = attn_out.size()
        attn_out = attn_out.permute(0, 2, 1, 3).contiguous().view(B, L, self.embed_size)
        return self.combine_heads(attn_out)


# Prueba rápida
test_tokens = torch.randint(0, len(vocab), (2, max_len))
test_mask   = (test_tokens != 0).long()
test_emb    = tpe(test_tokens)
mha = MultiHeadAttention(emb_dim, num_heads=8)
out = mha(test_emb, test_mask)
print(f'Input:  {test_emb.shape}  ->  Output MHA: {out.shape}')

### 4.3 Transformer Block (Encoder)

For classification, we only need the **encoder**. It combines:
1. Multi-Head Self-Attention
2. Add & Norm (residual connection + Layer Norm)
3. Feed-Forward Network (with **GELU** instead of ReLU)
4. Add & Norm

**Why GELU?** It is smoother than ReLU and is the standard activation in BERT, GPT-2, and most modern Transformers. Empirically, it converges better on NLP tasks.

In [ ]:
class TransformerBlock(nn.Module):
    """Bloque Encoder: MHA -> Add&Norm -> FFN -> Add&Norm"""

    def __init__(self, emb_dim: int, num_heads: int = 8, ff_dim: int = 512, dropout: float = 0.2):
        super().__init__()
        self.mhatt       = MultiHeadAttention(emb_dim, num_heads)
        self.mhatt_drop  = nn.Dropout(dropout)
        self.layer_norm1 = nn.LayerNorm(emb_dim)

        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.GELU(),           # Más suave que ReLU, estándar en transformers modernos
            nn.Dropout(dropout),
            nn.Linear(ff_dim, emb_dim)
        )
        self.ffn_drop    = nn.Dropout(dropout)
        self.layer_norm2 = nn.LayerNorm(emb_dim)

    def forward(self, x, mask=None):
        # Sub-capa 1: Atención + residual
        attn = self.mhatt_drop(self.mhatt(x, mask))
        x    = self.layer_norm1(x + attn)
        # Sub-capa 2: FFN + residual
        ffn  = self.ffn_drop(self.ffn(x))
        x    = self.layer_norm2(x + ffn)
        return x


tb  = TransformerBlock(emb_dim, num_heads=8)
out = tb(test_emb, test_mask)
print(f'Output TransformerBlock: {out.shape}')  # (2, max_len, emb_dim)

## 5. Full Classifier (LightningModule)

We assemble:
1. **TokenAndPosEmbedding** - tokens + position
2. **TransformerBlock** - attention mechanism
3. **Global Average Pooling** - averages over the sequence dimension before the classifier
4. **Dense classifier** - class logits

**Global Average Pooling** is more efficient than flattening the sequence (`nn.Flatten`) and more robust to padding.

In [ ]:
pip install pytorch-lightning torchmetrics

In [ ]:
import torch.nn.functional as F
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy


class LanguageIdentificationTransformer(LightningModule):
    """
    Encoder-only Transformer for language identification.
    Dataset: papluca/language-identification (20 classes).
    """

    def __init__(self, max_len, vocab_size, num_classes,
                 emb_dim=128, num_heads=8, ff_dim=512, dropout=0.2, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr

        self.embeddings  = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
        self.transformer = TransformerBlock(emb_dim, num_heads, ff_dim, dropout)

        # Classifier with implicit Global Average Pooling (mean over L)
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
            nn.LogSoftmax(dim=-1)
        )

        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc   = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc  = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, input_ids, attention_mask=None):
        x = self.embeddings(input_ids)           # (B, L, D)
        x = self.transformer(x, attention_mask)  # (B, L, D)
        x = x.mean(dim=1)                        # (B, D) — Global Average Pooling
        return self.classifier(x)                # (B, num_classes)

    def _shared_step(self, batch):
        ids, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        logits = self(ids, mask)
        loss   = F.nll_loss(logits, y)
        preds  = torch.argmax(logits, dim=-1)
        return loss, preds, y

    def training_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.train_acc(preds, y)
        self.log('train-loss', loss,           on_epoch=True, prog_bar=True)
        self.log('train-acc',  self.train_acc, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.val_acc(preds, y)
        self.log('val-loss', loss,         on_epoch=True, prog_bar=True)
        self.log('val-acc',  self.val_acc, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        loss, preds, y = self._shared_step(batch)
        self.test_acc(preds, y)
        self.log('test-acc', self.test_acc)

    def predict_step(self, batch, batch_idx):
        ids, mask = batch['input_ids'], batch['attention_mask']
        return torch.exp(self(ids, mask))  # log-probs -> probs

    def configure_optimizers(self):
        # AdamW + CosineAnnealing: standard for transformers
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]


model = LanguageIdentificationTransformer(
    max_len=max_len,
    vocab_size=len(vocab),
    num_classes=lang_dataset.num_classes,
    emb_dim=emb_dim,
    num_heads=8,
    ff_dim=512,
    dropout=0.2,
    lr=1e-3
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 6. Training

In [ ]:
tb_logger  = TensorBoardLogger('tb_logs', name='transformer_lang_id')
early_stop = EarlyStopping(monitor='val-loss', patience=3, mode='min', verbose=True)

trainer = Trainer(
    max_epochs=20,
    logger=tb_logger,
    callbacks=[early_stop],
    log_every_n_steps=10,
    accelerator='auto',
    devices=1
)

trainer.fit(model, train_loader, val_loader)

In [ ]:
import os
os.makedirs('tb_logs', exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir tb_logs

## 7. Test Evaluation

In [ ]:
model.eval()
test_results = trainer.test(model, dataloaders=test_loader)
print(f"\nTest Accuracy: {test_results[0]['test-acc']:.4f}")

## 8. Predictions and Analysis

In [ ]:
predictions  = trainer.predict(model, dataloaders=test_loader)
all_probs    = torch.cat(predictions, dim=0).numpy()  # (N, num_classes)
predicted_ids = np.argmax(all_probs, axis=1)
true_ids      = [sample['y'].item() for sample in test_dataset]

id_to_lang       = lang_dataset.id_2_class_map
true_labels      = [id_to_lang[t] for t in true_ids]
predicted_labels = [id_to_lang[p] for p in predicted_ids]

original_texts = [
    lang_dataset.dataset[test_dataset.indices[i]]['text']
    for i in range(len(test_dataset))
]

predictions_df = pd.DataFrame({
    'text':            original_texts,
    'true_label':      true_labels,
    'predicted_label': predicted_labels,
    'correct':         [t == p for t, p in zip(true_labels, predicted_labels)],
    'confidence':      np.max(all_probs, axis=1)
})

acc = predictions_df['correct'].mean()
print(f'Test accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'Correctas:        {predictions_df["correct"].sum()}')
print(f'Incorrectas:      {(~predictions_df["correct"]).sum()}')
predictions_df.head(10)

### 8.1 Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

class_labels = sorted(list(set(true_labels + predicted_labels)))
cm = confusion_matrix(true_labels, predicted_labels, labels=class_labels)

fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels).plot(
    cmap='Blues', ax=ax, xticks_rotation='vertical', colorbar=True
)
ax.set_title('Confusion Matrix - Transformer (Language Identification)', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix_transformer.png', dpi=120)
plt.show()

**Interpretation:**
- The main diagonal represents correct classifications.
- Off-diagonal cells indicate confusion between languages.
- Languages with non-Latin scripts (Arabic, Chinese, Japanese) tend to be the most problematic for this word-based tokenizer.

### 8.2 Per-Class Metrics

In [ ]:
report    = classification_report(true_labels, predicted_labels, output_dict=True)
report_df = pd.DataFrame(report).T
report_df = report_df[report_df.index.isin(class_labels)].sort_values('f1-score', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
x     = range(len(report_df))
width = 0.25
ax.bar([i - width for i in x], report_df['precision'], width=width, label='Precision', color='steelblue')
ax.bar([i         for i in x], report_df['recall'],    width=width, label='Recall',    color='coral')
ax.bar([i + width for i in x], report_df['f1-score'],  width=width, label='F1-score',  color='seagreen')
ax.set_xticks(list(x))
ax.set_xticklabels(report_df.index, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Precision, Recall, and F1-score by Language - Transformer', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=120)
plt.show()

print('Worst 5 languages by F1-score:')
print(report_df[['precision', 'recall', 'f1-score']].tail(5).to_string())

### 8.3 Most Confused Language Pairs

In [ ]:
errors_df = predictions_df[~predictions_df['correct']].copy()
conf_pairs = (errors_df.groupby(['true_label', 'predicted_label']).size()
              .reset_index(name='count')
              .sort_values('count', ascending=False)
              .head(15))

fig, ax = plt.subplots(figsize=(12, 5))
labels = [f"{r['true_label']} -> {r['predicted_label']}" for _, r in conf_pairs.iterrows()]
ax.barh(labels, conf_pairs['count'], color='salmon', edgecolor='white')
ax.set_title('Top 15 confused language pairs (true -> prediction)', fontsize=13)
ax.set_xlabel('Number of Errors')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('confusion_pairs.png', dpi=120)
plt.show()
print(conf_pairs.to_string(index=False))

### 8.4 Model Confidence Analysis

In [ ]:
correct_conf   = predictions_df[predictions_df['correct']]['confidence']
incorrect_conf = predictions_df[~predictions_df['correct']]['confidence']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(correct_conf,   bins=40, alpha=0.7, color='seagreen', label='Correct')
axes[0].hist(incorrect_conf, bins=40, alpha=0.7, color='salmon',   label='Incorrect')
axes[0].set_title('Model Confidence Distribution')
axes[0].set_xlabel('Maximum Predicted Probability')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)

conf_by_lang = predictions_df.groupby('true_label')['confidence'].mean().sort_values()
axes[1].barh(conf_by_lang.index, conf_by_lang.values, color='steelblue')
axes[1].axvline(conf_by_lang.mean(), color='red', linestyle='--',
                label=f'Media: {conf_by_lang.mean():.3f}')
axes[1].set_title('Average Confidence by Language')
axes[1].set_xlabel('Average confidence')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('confidence_analysis.png', dpi=120)
plt.show()

print(f'Average confidence on correct predictions:   {correct_conf.mean():.4f}')
print(f'Average confidence on incorrect predictions: {incorrect_conf.mean():.4f}')

**Note:** A well-calibrated model has high confidence on correct predictions and low confidence on incorrect ones. A wide gap between both distributions indicates good calibration.

### 8.5 Qualitative Error Examples

In [ ]:
print('=== High-confidence errors (the model was confident but wrong) ===')
high_conf_errors = predictions_df[
    (~predictions_df['correct']) & (predictions_df['confidence'] > 0.7)
].head(5)
for _, row in high_conf_errors.iterrows():
    print(f"\nText:       '{row['text'][:120]}...'")
    print(f"True:       {row['true_label']}")
    print(f"Predicted:  {row['predicted_label']} (confidence: {row['confidence']:.3f})")
    print('-' * 80)

print('\n=== Correct predictions with LOW confidence ===')
low_conf_correct = predictions_df[
    (predictions_df['correct']) & (predictions_df['confidence'] < 0.5)
].head(5)
for _, row in low_conf_correct.iterrows():
    print(f"\nText:    '{row['text'][:120]}...'")
    print(f"Language: {row['true_label']} (confidence: {row['confidence']:.3f})")
    print('-' * 80)

## 9. LSTM vs Transformer Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Feature': [
        'Architecture',
        'Trainable parameters',
        'Dataset',
        'Max seq. length',
        'Tokenizer',
        'Optimizer',
        'Test Accuracy (referencia LSTM)',
        'Sequence processing',
        'Context capture',
        'FFN activation',
        'Classification pooling'
    ],
    'LSTM': [
        'Bidirectional LSTM (2 layers)',
        '~1.5M',
        'papluca/language-identification',
        '128 tokens',
        'Simple word-level',
        'Adam',
        '~90%',
        'Sequential (recurrent)',
        'Local + memoria recurrente',
        'ReLU',
        'Last hidden state'
    ],
    'Transformer': [
        'Encoder-only (1 bloque)',
        f'{total_params:,}',
        'papluca/language-identification',
        '128 tokens',
        'Simple word-level (identical)',
        'AdamW + CosineAnnealing',
        'Ver celda 7',
        'Parallel (all tokens at once)',
        'Global (full attention)',
        'GELU',
        'Global Average Pooling'
    ]
})

comparison_df.style.set_properties(**{'text-align': 'left'})

## 10. Conclusions and Findings

### About the dataset
- **Perfectly balanced:** 20 languages with the same number of samples. No special sampling techniques are required.
- **Short texts:** Most are under 200 characters. With `max_length=128` tokens, the linguistic signal is captured well.
- **Non-Latin scripts** (Arabic, Chinese, Japanese, Hindi, Russian) are harder for the simple tokenizer, which collapses many characters into spaces. A BPE or character-level tokenizer would be more suitable.

### About the Transformer architecture
- **Global attention is advantageous for language identification:** the task relies on recognizing patterns distributed across the vocabulary. Attention can focus directly on discriminative words regardless of position, without gradually propagating information as in an LSTM.
- **Global Average Pooling > Flatten:** more robust to padding. It averages real token representations and effectively ignores padding, unlike `nn.Flatten`, which treats every token equally.
- **GELU over ReLU:** smoother gradients, no dead neurons, and empirically better convergence in NLP.
- **AdamW + CosineAnnealing:** a standard Transformer combination; decoupled weight decay helps prevent overfitting and cosine scheduling smoothly reduces the learning rate near the end.

### LSTM vs Transformer comparison
- Both models use the same dataset, tokenizer, and `max_length`, so the comparison is direct and fair.
- The Transformer processes all tokens **in parallel**, while the LSTM processes them **sequentially**, which is a clear GPU advantage.
- On short sequences like the ones in this dataset, the architectural difference is usually modest. The Transformer advantage becomes more pronounced on longer sequences or when long-range dependencies matter.
- Errors from both models tend to concentrate on the **same problematic languages**, suggesting the difficulty comes from the tokenizer rather than the architecture.

### Possible improvements
- Use a subword tokenizer (BPE, SentencePiece) to better handle non-Latin scripts.
- Stack 2-4 Transformer blocks for greater representational capacity.
- Use multilingual pretrained embeddings (`mBERT`, `XLM-RoBERTa`) as a starting point.
- Explore learnable `PositionalEncoding` instead of sinusoidal encoding (already available in the code with `PosEncodingType.LEARNABLE`).